# Simulation: Conditional Turning Bands

This notebook continues the geostatistical workflow from [3-estimation.ipynb](./3-estimation.ipynb). It uses the composites and variogram created in [1-populate-workspace.ipynb](./1-populate-workspace.ipynb) to run conditional simulation with Seequent Evo Compute.

Simulation is restricted to the Early Diorite region: only Early Diorite block-model cells are active in the regular masked 3-D target grid, and only Early Diorite composites are used as conditioning samples. The pointset's `Geological_model` must be a category attribute without nulls; declustering weights must also be valid.

Summary statistics, quantiles, cutoff probabilities, and saved realisations are written to the masked grid, not directly to the estimation block model. Running the creation and simulation cells mutates the selected workspace; these cells are not a dry run. Use a new grid name if the domain changes, preserving previous results.

In [ ]:
from dotenv import dotenv_values
from helpers import create_masked_grid_from_block_model, variogram_structure

import evo.objects.typed as objs
from evo.compute import tasks
from evo.compute.tasks.geostatistics import conditioned_simulator as sim
from evo.notebooks import ServiceManagerWidget

%load_ext evo.widgets

## Authenticate and load the input objects

Select the workspace populated by the first geostatistics notebook. The input objects are references on Evo; their full datasets are not downloaded just to configure the compute task.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(**dotenv_values()).login()

In [ ]:
comps = await objs.object_from_path(evo_connection, "WP comps.json")
block_model = await objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

## Prepare a normal-score variogram

Conditional simulation requires a **normal-score** variogram with unit sill. Use `variogram_structure` from `helpers.py` to recreate the supplied model: nugget 0.1 and two spherical structures with contributions 0.1 and 0.8. Their major, semi-major, and minor ranges are (12, 10, 100) and (375, 265, 100), respectively. Both share dip azimuth 70 degrees, dip 15 degrees, and pitch 0 degrees.

The supplied parameters are used directly, without scaling the estimation variogram. The first structure's minor range exceeds its major and semi-major ranges; these values are preserved as supplied. Fit and validate the model against normal-score-transformed samples before production use.

In [ ]:
variogram_data = objs.VariogramData(
    name="Cu_pct NScore Variogram",
    description="Cu_pct NScore Variogram",
    sill=1.0,
    nugget=0.1,
    is_rotation_fixed=True,
    modelling_space="normalscore",
    data_variance=1.0,
    attribute="Cu_pct NS",
    structures=[
        variogram_structure(
            "spherical",
            contribution=0.1,
            major=12.0,
            semi_major=10.0,
            minor=100.0,
            rotation=(70.0, 15.0, 0.0),
        ),
        variogram_structure(
            "spherical",
            contribution=0.8,
            major=375.0,
            semi_major=265.0,
            minor=100.0,
            rotation=(70.0, 15.0, 0.0),
        ),
    ],
)


# Create the variogram object in the Evo workspace. If this variogram already exists, retrieve it instead.
try:
    normal_score_variogram = await objs.Variogram.create(evo_connection, variogram_data)
except Exception:
    normal_score_variogram = await objs.object_from_path(evo_connection, variogram_data.name + ".json")

## Create the Early Diorite masked grid

Use the block model's `Geological model` attribute to activate only Early Diorite cells. The masked grid preserves the block model's origin, rotation, CRS, extent, and block size, so every active cell corresponds to one geological block. Other and missing geology values remain inactive.

The grid is named `WP Early Diorite Cu_pct Simulation Grid`, separately from the old unmasked grid. An existing grid is reused only if its geometry and mask match. Confirm the selected organization and workspace and the input coordinate systems before running this cell.

In [ ]:
block_geology = await block_model.to_dataframe(columns=["i", "j", "k", "Geological model"])
early_diorite_blocks = block_geology.loc[block_geology["Geological model"].eq("Early Diorite")]
grid_data = create_masked_grid_from_block_model(
    block_model,
    early_diorite_blocks,
    name="WP Early Diorite Cu_pct Simulation Grid",
    description="Copper simulation restricted to Early Diorite blocks",
)


grid = await objs.RegularMasked3DGrid.create(evo_connection, grid_data)
grid

## Configure and run conditional simulation

Generate an ensemble of 100 realisations, save 10 realizations for the purpose of point scale validation and retain all 100 at the blocksale. Additionally calculate the P10, P50 and P90 quantiles along with the probability of being above a 0.5% copper cutoff. Include `perform_validation=True` to generate the dashboard of validation plots and statististics.

In [ ]:
search = tasks.SearchNeighborhood(
    ellipsoid=normal_score_variogram.get_ellipsoid().scaled(2.0), max_samples=20, min_samples=1
)

early_diorite_filter = tasks.Filter(
    where=tasks.FilterCondition(
        attribute=comps.attributes["Geological_model"],
        operator="in",
        values=["Early Diorite"],
    ),
)

simulation_params = sim.ConSimParameters(
    source_object=comps,
    source_attribute=comps.attributes["Cu_pct"],
    source_filter=early_diorite_filter,
    target_object=grid,
    variogram_model=normal_score_variogram,
    neighborhood=search,
    kriging_method="simple",
    distribution=sim.DistributionParams(weights=comps.attributes["IDW declus"]),
    block_discretization=sim.BlockDiscretization(nx=2, ny=2, nz=2),
    number_of_lines=500,
    number_of_simulations=100,
    number_of_simulations_to_save=10,
    random_seed=38239342,
    location_wise_quantiles=[0.1, 0.5, 0.9],
    probability_above_cutoff=[0.5],
    perform_validation=True,
)

Now run the simulation

In [ ]:
result = await tasks.run(evo_connection, simulation_params, preview=True)

Query the `dashboard_url` to launch the dashboard in your browser

In [ ]:
result.dashboard_url